# 05 - SHAP Explainability

Interpret the tuned XGBoost model using SHAP (SHapley Additive exPlanations).
Every plot is saved to `outputs/figures/` and prefixed `05_`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

MODELS_PATH  = Path('../outputs/models')
FIGURES_PATH = Path('../outputs/figures')
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

## 1. Load model and test data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = joblib.load('../data/processed/splits.pkl')
scaler = joblib.load(MODELS_PATH / 'scaler.pkl')
X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)
model = joblib.load(MODELS_PATH / 'best_xgb.pkl')
print(f'Test samples: {X_test_scaled_df.shape[0]:,}')
print(f'Fraud cases:  {y_test.sum():,}')

## 2. SHAP TreeExplainer

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_scaled_df)

print(f'shap_values shape : {shap_values.shape}')
print(f'Expected value    : {explainer.expected_value:.4f}')

## 3. Global summary - bar chart

Mean absolute SHAP value per feature across all test transactions.

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled_df, plot_type='bar', show=False)
plt.title('Mean |SHAP| - global feature importance')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '05_shap_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Beeswarm - feature effect direction

Each dot is one transaction. Colour = feature value; x-axis = impact on model output.

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_scaled_df, show=False)
plt.title('SHAP beeswarm - feature effect direction')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '05_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Waterfall - single fraud prediction

Breaks down one fraud transaction into per-feature contributions.

In [ ]:
fraud_indices = np.where(y_test.values == 1)[0]
idx           = fraud_indices[0]

shap.waterfall_plot(
    shap.Explanation(
        values       = shap_values[idx],
        base_values  = explainer.expected_value,
        data         = X_test_scaled_df.iloc[idx],
        feature_names= X_test_scaled_df.columns.tolist(),
    ),
    show=False,
)
plt.title(f'Waterfall - why transaction #{idx} was flagged as fraud')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '05_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Force plot - first 50 test samples

Interactive HTML force plot saved to disk.

In [ ]:
force_plot = shap.force_plot(
    explainer.expected_value,
    shap_values[:50],
    X_test_scaled_df.iloc[:50],
    show=False,
)
shap.save_html(str(FIGURES_PATH / '05_shap_force_plot.html'), force_plot)
print('Saved: outputs/figures/05_shap_force_plot.html')

## 7. Mean |SHAP| feature ranking

In [ ]:
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test_scaled_df.columns
).sort_values(ascending=False)

print('Top 10 features by mean |SHAP|:')
print(mean_shap.head(10).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
mean_shap.head(15).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 15 features - mean |SHAP|')
ax.set_xlabel('Feature')
ax.set_ylabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig(FIGURES_PATH / '05_shap_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Dependence plot - V14

Shows the relationship between V14 values and fraud risk.

In [ ]:
plt.figure(figsize=(10, 6))
shap.dependence_plot('V14', shap_values, X_test_scaled_df, show=False)
plt.title('SHAP dependence plot - V14 vs fraud risk')
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'shap_dependence_v14.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test_scaled_df.columns
).sort_values(ascending=False)

print('=== SHAP GLOBAL SUMMARY ===')
print(f'
Top 5 fraud signals:')
for feat, score in mean_shap.head(5).items():
    print(f'  {feat:15s}  mean |SHAP|: {score:.4f}')

print(f'
Next: notebook 06 - threshold tuning')
print('At what fraud probability do we block a transaction?')